# Olist E-Commerce: Gold Layer
**Dataset**: [Brazilian E-commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)   
**Layer**: Gold: Business-ready analytics tables   
**Tools Used**: Databricks, Spark SQL, Delta Lake  

## Overview
This notebook builds the gold layer of a Medallion Architecture pipeline on the Olist e-commerce dataset. It transforms cleaned silver tables into star-schema fact tables, dimension tables, and pre-aggregated business metrics ready for BI tooling.

The Olist dataset covers ~100,000 orders placed across Brazil between 2016 and 2018, spanning 9 source tables including customers, sellers, products, orders, payments, and reviews.

## Components
### Fact Tables
- [`gold_fact_order_items`](#fact_order_items) — Item-level transaction grain
- [`gold_fact_orders`](#fact_orders) — Order-level transaction grain

### Dimension Tables
- [`gold_dim_customers`](#dim_customers) — Customer master data
- [`gold_dim_products`](#dim_products) — Product catalogue
- [`gold_dim_sellers`](#dim_sellers) — Seller master data
- [`gold_dim_dates`](#dim_dates) — Date intelligence calendar

### Aggregate Tables
- [`gold_agg_daily_sales`](#agg_daily) — Daily sales summary
- [`gold_agg_monthly_sales`](#agg_monthly) — Monthly sales summary with MoM metrics
- [`gold_agg_product_performance`](#agg_product) — Product-level revenue and delivery metrics
- [`gold_agg_top_sellers`](#agg_sellers) — Seller rankings and performance scorecards
- [`gold_agg_customer_lifespan`](#agg_customer) — Customer lifetime value and segmentation
- [`gold_agg_regional_sales`](#agg_regional) — Revenue and volume by Brazilian region and state

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT 
    table_name, 
    column_name, 
    data_type
FROM information_schema.columns
WHERE table_schema = 'default'
  AND table_name IN ('silver_customers', 'silver_geolocation', 'silver_order_items', 'silver_order_payments', 'silver_order_reviews', 'silver_orders', 'silver_products', 'silver_sellers')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
silver_customers,customer_id,STRING
silver_customers,customer_unique_id,STRING
silver_customers,customer_zip_code_prefix,INT
silver_customers,customer_city,STRING
silver_customers,customer_state,STRING
silver_customers,ingestion_timestamp,TIMESTAMP
silver_customers,customer_city_cleaned,STRING
silver_customers,zip_code_valid,BOOLEAN
silver_customers,state_valid,BOOLEAN
silver_customers,customer_region,STRING


## Fact Tables

Fact tables record business events at a specific grain. Each row represents one occurrence of that event: either an individual order line item or a complete order. Fact tables are the primary source for revenue, volume, and delivery metrics and are 
designed to be joined to dimension tables using shared key columns.

All fact tables in this layer:
- Are sourced exclusively from validated silver tables
- Carry pre-calculated date difference columns to avoid repeated computation downstream
- Contain a `created_at` audit timestamp for pipeline lineage tracking

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_order_items AS

SELECT
    --Fact Key
    CONCAT(oi.order_id, '_', oi.order_item_id) AS order_item_key,
    --Dimension Keys
    oi.order_id,
    oi.product_id,
    oi.seller_id,
    o.customer_id,
    --Date Keys
    o.order_purchase_timestamp AS order_date_key,
    oi.shipping_limit_date AS shipping_date_key,
    o.order_delivered_customer_timestamp AS delivery_date_key,  
    -- Order Context
    oi.order_item_id,
    o.order_status_cleaned AS order_status,
    --Measures
    1 AS item_quantity, --for order items, each row is one item
    oi.price AS item_price,
    oi.freight_value AS item_freight,
    ROUND(oi.total_item_value, 2) AS item_total_value,
    --Dates
    o.order_purchase_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    oi.shipping_limit_date,
    --Date Calculations
    DATEDIFF(o.order_delivered_customer_date, o.order_purchase_date) AS days_to_deliver,
    DATEDIFF(o.order_estimated_delivery_date, o.order_delivered_customer_date) AS delivery_vs_estimate_days,
    --Flags
    o.delivery_performance,
    o.timeline_flag,
    CASE WHEN o.order_status_cleaned = 'Delivered' THEN TRUE ELSE FALSE END AS is_delivered,
    CASE WHEN o.order_status_cleaned = 'Canceled' THEN TRUE ELSE FALSE END AS is_canceled,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at
FROM silver_order_items oi
INNER JOIN silver_orders o ON
    oi.order_id = o.order_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_counts AS (
    SELECT COUNT(*) AS silver_row_count
    FROM silver_order_items
),
gold_counts AS (
    SELECT
        COUNT(*)                                                        AS gold_row_count,
        COUNT(DISTINCT order_item_key)                                  AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT order_item_key)                       AS duplicate_keys,
        COUNT(DISTINCT order_id)                                        AS distinct_orders,
        COUNT(DISTINCT product_id)                                      AS distinct_products,
        COUNT(DISTINCT seller_id)                                       AS distinct_sellers,
        SUM(CASE WHEN order_item_key IS NULL THEN 1 ELSE 0 END)        AS null_keys,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)              AS null_order_ids,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)            AS null_product_ids,
        SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END)             AS null_seller_ids,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)           AS null_customer_ids,
        SUM(CASE WHEN item_price <= 0 THEN 1 ELSE 0 END)               AS non_positive_prices,
        SUM(CASE WHEN item_freight < 0 THEN 1 ELSE 0 END)              AS negative_freight,
        SUM(CASE WHEN item_total_value <= 0 THEN 1 ELSE 0 END)         AS non_positive_total_values,
        SUM(CASE WHEN days_to_deliver < 0 THEN 1 ELSE 0 END)           AS negative_delivery_days,
        SUM(CASE WHEN is_delivered = TRUE
                  AND days_to_deliver IS NULL THEN 1 ELSE 0 END)        AS delivered_missing_days,
        SUM(CASE WHEN delivery_performance NOT IN
                ('Early', 'On Time', 'Late', 'Pending')
                 OR delivery_performance IS NULL THEN 1 ELSE 0 END)     AS invalid_delivery_flags,
        ROUND(MIN(item_price), 2)                                       AS min_price,
        ROUND(MAX(item_price), 2)                                       AS max_price,
        ROUND(AVG(item_price), 2)                                       AS avg_price,
        ROUND(SUM(item_total_value), 2)                                 AS total_revenue
    FROM gold_fact_order_items
)
SELECT
    -- Row parity
    s.silver_row_count,
    g.gold_row_count,
    g.gold_row_count - s.silver_row_count                              AS row_difference,

    -- Key integrity
    g.distinct_keys,
    g.duplicate_keys,
    g.null_keys,

    -- Referential counts
    g.distinct_orders,
    g.distinct_products,
    g.distinct_sellers,

    -- NULL checks on FK columns
    g.null_order_ids,
    g.null_product_ids,
    g.null_seller_ids,
    g.null_customer_ids,

    -- Business rule checks
    g.non_positive_prices,
    g.negative_freight,
    g.non_positive_total_values,
    g.negative_delivery_days,
    g.delivered_missing_days,
    g.invalid_delivery_flags,

    -- Sanity metrics
    g.min_price,
    g.max_price,
    g.avg_price,
    g.total_revenue,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != s.silver_row_count THEN 'FAIL: Row count mismatch vs silver'
        WHEN g.duplicate_keys > 0                   THEN 'FAIL: Duplicate order_item_keys'
        WHEN g.null_keys > 0                        THEN 'FAIL: NULL order_item_keys'
        WHEN g.null_order_ids > 0                   THEN 'FAIL: NULL order_ids'
        WHEN g.null_product_ids > 0                 THEN 'FAIL: NULL product_ids'
        WHEN g.null_seller_ids > 0                  THEN 'FAIL: NULL seller_ids'
        WHEN g.non_positive_prices > 0              THEN 'FAIL: Non-positive item prices'
        WHEN g.negative_freight > 0                 THEN 'FAIL: Negative freight values'
        WHEN g.non_positive_total_values > 0        THEN 'FAIL: Non-positive total values'
        WHEN g.negative_delivery_days > 0           THEN 'WARN: Negative delivery days detected'
        WHEN g.invalid_delivery_flags > 0           THEN 'WARN: Unexpected delivery_performance values'
        ELSE                                             'PASS'
    END AS validation_result

FROM silver_counts s, gold_counts g;

silver_row_count,gold_row_count,row_difference,distinct_keys,duplicate_keys,null_keys,distinct_orders,distinct_products,distinct_sellers,null_order_ids,null_product_ids,null_seller_ids,null_customer_ids,non_positive_prices,negative_freight,non_positive_total_values,negative_delivery_days,delivered_missing_days,invalid_delivery_flags,min_price,max_price,avg_price,total_revenue,validation_result
112650,112650,0,112650,0,0,98666,32951,3095,0,0,0,0,0,0,0,0,0,2454,0.85,6735.0,120.65,1.584355324E7,WARN: Unexpected delivery_performance values


In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_orders AS

SELECT
    --Dimensions Keys
    o.order_id,
    o.customer_id,
    --Date Keys
    o.order_purchase_timestamp AS order_date_key,
    o.order_delivered_customer_timestamp AS delivery_date_key, 
    -- Order Attributes
    o.order_status_cleaned AS order_status,
    -- Order Measures (aggregated from order_items)
    COALESCE(items.total_items, 0) AS total_items,
    COALESCE(items.unique_sellers, 0) AS unique_sellers,
    COALESCE(items.unique_products, 0) AS unique_products,
    COALESCE(items.subtotal, 0) AS order_subtotal,
    COALESCE(items.total_freight, 0) AS order_freight,
    ROUND(COALESCE(items.subtotal + items.total_freight, 0), 2) AS order_total_value,
    -- Payment Measures
    COALESCE(pay.total_payment, 0) AS total_payment_received,
    COALESCE(pay.payment_types, 0) AS distinct_payment_types,
    COALESCE(pay.total_installments, 0) AS total_installments,
    -- Review Measures
    rev.review_score AS review_score,
    --Date Calculations
    DATEDIFF(o.order_delivered_customer_date, o.order_purchase_date) AS days_to_deliver,
    DATEDIFF(o.order_estimated_delivery_date, o.order_delivered_customer_date) AS delivery_vs_estimate_days,
    --Flags
    o.delivery_performance,
    o.timeline_flag,
    CASE WHEN o.order_status_cleaned = 'Delivered' THEN TRUE ELSE FALSE END AS is_delivered,
    CASE WHEN o.order_status_cleaned = 'Canceled' THEN TRUE ELSE FALSE END AS is_canceled,
    CASE WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN TRUE ELSE FALSE END AS delivered_on_time,
    CASE WHEN ABS(total_payment_received - order_total_value) > 0.1 AND order_status NOT IN ('Canceled', 'Unavailable') THEN 'Violation' ELSE 'Valid' END AS payment_to_total_discrepancies, --An acceptable threshold for BRL in a BI report is typically R$ 0.10 for general reconciliation.
    --Audit
    CURRENT_TIMESTAMP() AS created_at
FROM silver_orders o
JOIN (
    SELECT 
        order_id,
        COUNT(*) AS total_items,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        COUNT(DISTINCT product_id) AS unique_products,
        ROUND(SUM(price), 2) AS subtotal,
        ROUND(SUM(freight_value), 2) AS total_freight
    FROM silver_order_items
    GROUP BY order_id
) items ON o.order_id = items.order_id
JOIN (
    SELECT 
        order_id,
        ROUND(SUM(payment_value), 2) AS total_payment,
        COUNT(DISTINCT payment_type) AS payment_types,
        ROUND(SUM(payment_installments), 2) AS total_installments
    FROM silver_order_payments
    GROUP BY order_id
) pay ON o.order_id = pay.order_id
LEFT JOIN (
    SELECT 
        order_id,
        ROUND(AVG(review_score_cleaned), 2) AS review_score
    FROM silver_order_reviews
    GROUP BY order_id
) rev ON o.order_id = rev.order_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_eligible AS (
    -- Counts only orders that have both item and payment records, matching INNER JOIN logic in gold_fact_orders.
    SELECT
        COUNT(*)                                                            AS silver_row_count,
        SUM(CASE WHEN i.order_id IS NOT NULL
                  AND p.order_id IS NOT NULL THEN 1 ELSE 0 END)            AS eligible_row_count,
        SUM(CASE WHEN i.order_id IS NULL
                  OR p.order_id IS NULL THEN 1 ELSE 0 END)                 AS intentionally_excluded
    FROM silver_orders o
    LEFT JOIN (SELECT DISTINCT order_id FROM silver_order_items)    i ON o.order_id = i.order_id
    LEFT JOIN (SELECT DISTINCT order_id FROM silver_order_payments) p ON o.order_id = p.order_id
),
gold_counts AS (
    SELECT
        COUNT(*)                                                            AS gold_row_count,
        COUNT(DISTINCT order_id)                                            AS distinct_orders,
        COUNT(*) - COUNT(DISTINCT order_id)                                 AS duplicate_orders,
        COUNT(DISTINCT customer_id)                                         AS distinct_customers,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)                  AS null_order_ids,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)               AS null_customer_ids,
        SUM(CASE WHEN order_total_value < 0 THEN 1 ELSE 0 END)             AS negative_order_values,
        SUM(CASE WHEN order_total_value = 0
                  AND order_status NOT IN ('Canceled', 'Unavailable')
                  THEN 1 ELSE 0 END)                                        AS unexpected_zero_values,
        SUM(CASE WHEN total_payment_received < 0 THEN 1 ELSE 0 END)        AS negative_payments,
        SUM(CASE WHEN days_to_deliver < 0 THEN 1 ELSE 0 END)               AS negative_delivery_days,
        SUM(CASE WHEN is_delivered = TRUE
                  AND days_to_deliver IS NULL THEN 1 ELSE 0 END)            AS delivered_missing_days,
        SUM(CASE WHEN is_delivered = TRUE
                  AND is_canceled = TRUE THEN 1 ELSE 0 END)                 AS conflicting_status_flags,
        SUM(CASE WHEN payment_to_total_discrepancies
                  = 'Violation' THEN 1 ELSE 0 END)                          AS payment_violations,
        SUM(CASE WHEN review_score IS NOT NULL
                  AND (review_score < 1
                  OR review_score > 5) THEN 1 ELSE 0 END)                   AS invalid_review_scores,
        SUM(CASE WHEN total_items = 0 THEN 1 ELSE 0 END)                    AS orders_with_no_items,
        SUM(CASE WHEN total_payment_received = 0
                  AND order_status NOT IN ('Canceled', 'Unavailable')
                  THEN 1 ELSE 0 END)                                         AS unpaid_active_orders,
        COUNT(CASE WHEN review_score IS NOT NULL THEN 1 END)                AS orders_with_reviews,
        ROUND(AVG(order_total_value), 2)                                    AS avg_order_value,
        ROUND(MIN(order_total_value), 2)                                    AS min_order_value,
        ROUND(MAX(order_total_value), 2)                                    AS max_order_value,
        ROUND(SUM(order_total_value), 2)                                    AS total_revenue,
        ROUND(AVG(review_score), 2)                                         AS avg_review_score,
        ROUND(SUM(CASE WHEN delivered_on_time THEN 1 ELSE 0 END) * 100.0
              / NULLIF(SUM(CASE WHEN is_delivered THEN 1 ELSE 0 END), 0), 2) AS on_time_pct
    FROM gold_fact_orders
)
SELECT
    -- Row parity (compared against eligible silver rows, not total silver rows)
    s.silver_row_count,
    s.eligible_row_count,
    s.intentionally_excluded,
    g.gold_row_count,
    g.gold_row_count - s.eligible_row_count                                AS unexpected_row_difference,

    -- Key integrity
    g.distinct_orders,
    g.duplicate_orders,
    g.distinct_customers,

    -- NULL checks
    g.null_order_ids,
    g.null_customer_ids,

    -- Business rule checks
    g.negative_order_values,
    g.unexpected_zero_values,
    g.negative_payments,
    g.negative_delivery_days,
    g.delivered_missing_days,
    g.conflicting_status_flags,
    g.payment_violations,
    g.invalid_review_scores,
    g.orders_with_no_items,
    g.unpaid_active_orders,

    -- Sanity metrics
    g.orders_with_reviews,
    g.avg_order_value,
    g.min_order_value,
    g.max_order_value,
    g.total_revenue,
    g.avg_review_score,
    g.on_time_pct,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != s.eligible_row_count THEN 'FAIL: Unexpected row count vs eligible silver orders'
        WHEN g.duplicate_orders > 0                   THEN 'FAIL: Duplicate order_ids'
        WHEN g.null_order_ids > 0                     THEN 'FAIL: NULL order_ids'
        WHEN g.null_customer_ids > 0                  THEN 'FAIL: NULL customer_ids'
        WHEN g.negative_order_values > 0              THEN 'FAIL: Negative order values'
        WHEN g.negative_payments > 0                  THEN 'FAIL: Negative payment amounts'
        WHEN g.conflicting_status_flags > 0           THEN 'FAIL: Orders flagged both delivered and canceled'
        WHEN g.orders_with_no_items > 0               THEN 'FAIL: Orders with zero items'
        WHEN g.invalid_review_scores > 0              THEN 'FAIL: Review scores outside 1–5 range'
        WHEN g.payment_violations > 0                 THEN 'WARN: Payment reconciliation violations detected'
        WHEN g.unpaid_active_orders > 0               THEN 'WARN: Active orders with zero payment'
        WHEN g.unexpected_zero_values > 0             THEN 'WARN: Non-canceled orders with zero value'
        WHEN g.negative_delivery_days > 0             THEN 'WARN: Negative delivery days detected'
        ELSE                                               'PASS'
    END AS validation_result

FROM silver_eligible s, gold_counts g;
-- NOTE ON ROW DIFFERENCE: gold_fact_orders vs silver_orders (~776 rows): gold_fact_orders uses INNER JOINs on silver_order_items and silver_order_payments. This intentionally excludes any orders that have no corresponding item or payment records (i.e., canceled or flagged prior to transaction).

silver_row_count,eligible_row_count,intentionally_excluded,gold_row_count,unexpected_row_difference,distinct_orders,duplicate_orders,distinct_customers,null_order_ids,null_customer_ids,negative_order_values,unexpected_zero_values,negative_payments,negative_delivery_days,delivered_missing_days,conflicting_status_flags,payment_violations,invalid_review_scores,orders_with_no_items,unpaid_active_orders,orders_with_reviews,avg_order_value,min_order_value,max_order_value,total_revenue,avg_review_score,on_time_pct,validation_result
99441,98665,776,98665,0,98665,0,98665,0,0,0,0,0,0,0,0,257,0,0,0,97916,160.58,9.59,13664.08,1.584340978E7,4.11,93.23,WARN: Payment reconciliation violations detected


## Dimension Tables

These dimension tables provide the descriptive context for fact table metrics and are joined to fact tables using shared key columns and are designed to be stable reference data that changes infrequently.

All dimension tables in this layer:
- Use the natural key from the silver layer as the primary key
- Are enriched with geographic coordinates from `silver_geolocation` where applicable, using `AVG()` to resolve any residual zip code duplicates
- Carry `created_at` and `updated_at` audit timestamps



### Dimension_Customers

Customer Details

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_customers AS

SELECT
    -- Natural Keys
    c.customer_id,  -- Primary key
    c.customer_unique_id,  -- For linking customer history
    --Attributes
    c.customer_city_cleaned AS customer_city,
    c.customer_state,
    c.customer_region,
    c.customer_zip_code_prefix AS customer_zip_code,
    AVG(g.geolocation_lat_updated) AS latitude,
    AVG(g.geolocation_lng_updated) AS longitude,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_TIMESTAMP() AS updated_at
FROM silver_customers c  
LEFT JOIN silver_geolocation g ON
    c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
GROUP BY
    c.customer_id,
    c.customer_unique_id,
    c.customer_city_cleaned,
    c.customer_state,
    c.customer_region,
    c.customer_zip_code_prefix
ORDER BY customer_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_counts AS (
    SELECT COUNT(*) AS silver_row_count
    FROM silver_customers
),
gold_counts AS (
    SELECT
        COUNT(*)                                                            AS gold_row_count,
        COUNT(DISTINCT customer_id)                                         AS distinct_customer_ids,
        COUNT(*) - COUNT(DISTINCT customer_id)                              AS duplicate_customer_ids,
        COUNT(DISTINCT customer_unique_id)                                  AS distinct_unique_customers,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)               AS null_customer_ids,
        SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END)        AS null_unique_ids,
        SUM(CASE WHEN customer_city IS NULL THEN 1 ELSE 0 END)             AS null_cities,
        SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END)            AS null_states,
        SUM(CASE WHEN customer_region IS NULL THEN 1 ELSE 0 END)           AS null_regions,
        SUM(CASE WHEN customer_zip_code IS NULL THEN 1 ELSE 0 END)         AS null_zip_codes,
        SUM(CASE WHEN customer_zip_code < 1000
                  OR customer_zip_code > 99999 THEN 1 ELSE 0 END)          AS invalid_zip_codes,
        SUM(CASE WHEN customer_state NOT IN (
                'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA',
                'MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN',
                'RS','RO','RR','SC','SP','SE','TO')
                THEN 1 ELSE 0 END)                                          AS invalid_states,
        SUM(CASE WHEN customer_region NOT IN (
                'North', 'Northeast', 'Central-West', 'Southeast', 'South')
                THEN 1 ELSE 0 END)                                          AS invalid_regions,
        SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END)                  AS null_latitudes,
        SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END)                 AS null_longitudes,
        SUM(CASE WHEN latitude < -33.75
                  OR latitude > 5.27 THEN 1 ELSE 0 END)                    AS out_of_range_latitudes,
        SUM(CASE WHEN longitude < -73.99
                  OR longitude > -28.85 THEN 1 ELSE 0 END)                 AS out_of_range_longitudes,
        COUNT(DISTINCT customer_state)                                      AS distinct_states,
        COUNT(DISTINCT customer_region)                                     AS distinct_regions
    FROM gold_dim_customers
)
SELECT
    -- Row parity
    s.silver_row_count,
    g.gold_row_count,
    g.gold_row_count - s.silver_row_count                                  AS row_difference,

    -- Key integrity
    g.distinct_customer_ids,
    g.duplicate_customer_ids,
    g.distinct_unique_customers,

    -- NULL checks
    g.null_customer_ids,
    g.null_unique_ids,
    g.null_cities,
    g.null_states,
    g.null_regions,
    g.null_zip_codes,

    -- Domain validation
    g.invalid_zip_codes,
    g.invalid_states,
    g.invalid_regions,

    -- Geolocation checks
    g.null_latitudes,
    g.null_longitudes,
    g.out_of_range_latitudes,
    g.out_of_range_longitudes,

    -- Sanity metrics
    g.distinct_states,
    g.distinct_regions,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != s.silver_row_count     THEN 'FAIL: Row count mismatch vs silver'
        WHEN g.duplicate_customer_ids > 0               THEN 'FAIL: Duplicate customer_ids'
        WHEN g.null_customer_ids > 0                    THEN 'FAIL: NULL customer_ids'
        WHEN g.null_unique_ids > 0                      THEN 'FAIL: NULL customer_unique_ids'
        WHEN g.invalid_states > 0                       THEN 'FAIL: Invalid Brazilian state codes'
        WHEN g.invalid_regions > 0                      THEN 'FAIL: Invalid region values'
        WHEN g.invalid_zip_codes > 0                    THEN 'FAIL: Zip codes outside valid range'
        WHEN g.out_of_range_latitudes > 0               THEN 'WARN: Latitudes outside Brazilian bounds'
        WHEN g.out_of_range_longitudes > 0              THEN 'WARN: Longitudes outside Brazilian bounds'
        WHEN g.null_latitudes > 0                       THEN 'WARN: Customers with no latitude (zip not in geolocation)'
        WHEN g.distinct_regions != 5                    THEN 'WARN: Expected 5 distinct regions'
        ELSE                                                 'PASS'
    END AS validation_result

FROM silver_counts s, gold_counts g;

silver_row_count,gold_row_count,row_difference,distinct_customer_ids,duplicate_customer_ids,distinct_unique_customers,null_customer_ids,null_unique_ids,null_cities,null_states,null_regions,null_zip_codes,invalid_zip_codes,invalid_states,invalid_regions,null_latitudes,null_longitudes,out_of_range_latitudes,out_of_range_longitudes,distinct_states,distinct_regions,validation_result
99441,99441,0,99441,0,96096,0,0,0,0,0,0,0,0,0,279,279,0,0,27,5,WARN: Customers with no latitude (zip not in geolocation)


In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_products AS

SELECT
    -- Keys
    product_id,
    --Product Attributes
    product_category_updated AS product_category,
    product_weight_kg,
    product_length_cm,
    product_height_cm,
    product_width_cm,
    product_photo_quantity,
    --Calculated Attributes
    product_volume_cm3,
    CASE
        WHEN product_weight_kg > 10 THEN 'Heavy'
        WHEN product_weight_kg > 1 THEN 'Medium'
        ELSE 'Light'
    END AS weight_category,
    CASE
        WHEN product_volume_cm3 > 100000 THEN 'Large'
        WHEN product_volume_cm3 > 10000 THEN 'Medium'
        Else 'Small'
    END AS size_category,

    -- Audit
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_TIMESTAMP() AS updated_at
FROM silver_products;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_counts AS (
    SELECT COUNT(*) AS silver_row_count
    FROM silver_products
),
gold_counts AS (
    SELECT
        COUNT(*)                                                            AS gold_row_count,
        COUNT(DISTINCT product_id)                                          AS distinct_product_ids,
        COUNT(*) - COUNT(DISTINCT product_id)                               AS duplicate_product_ids,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)                AS null_product_ids,
        SUM(CASE WHEN product_category IS NULL THEN 1 ELSE 0 END)          AS null_categories,
        SUM(CASE WHEN product_category = 'Unknown' THEN 1 ELSE 0 END)      AS unknown_categories,
        SUM(CASE WHEN product_weight_kg IS NULL THEN 1 ELSE 0 END)         AS null_weights,
        SUM(CASE WHEN product_weight_kg <= 0 THEN 1 ELSE 0 END)            AS non_positive_weights,
        SUM(CASE WHEN product_length_cm IS NULL
                  OR product_height_cm IS NULL
                  OR product_width_cm IS NULL THEN 1 ELSE 0 END)           AS null_dimensions,
        SUM(CASE WHEN product_length_cm <= 0
                  OR product_height_cm <= 0
                  OR product_width_cm <= 0 THEN 1 ELSE 0 END)              AS non_positive_dimensions,
        SUM(CASE WHEN product_volume_cm3 IS NULL THEN 1 ELSE 0 END)        AS null_volumes,
        SUM(CASE WHEN product_volume_cm3 <= 0 THEN 1 ELSE 0 END)           AS non_positive_volumes,
        SUM(CASE WHEN product_photo_quantity < 0 THEN 1 ELSE 0 END)        AS negative_photo_counts,
        SUM(CASE WHEN weight_category NOT IN
                ('Light', 'Medium', 'Heavy') THEN 1 ELSE 0 END)            AS invalid_weight_categories,
        SUM(CASE WHEN size_category NOT IN
                ('Small', 'Medium', 'Large') THEN 1 ELSE 0 END)            AS invalid_size_categories,
        COUNT(DISTINCT product_category)                                    AS distinct_categories,
        ROUND(AVG(product_weight_kg), 3)                                   AS avg_weight_kg,
        ROUND(MAX(product_weight_kg), 3)                                   AS max_weight_kg,
        ROUND(AVG(product_volume_cm3), 2)                                  AS avg_volume_cm3
    FROM gold_dim_products
)
SELECT
    -- Row parity
    s.silver_row_count,
    g.gold_row_count,
    g.gold_row_count - s.silver_row_count                                  AS row_difference,

    -- Key integrity
    g.distinct_product_ids,
    g.duplicate_product_ids,
    g.null_product_ids,

    -- Category checks
    g.null_categories,
    g.unknown_categories,
    g.distinct_categories,

    -- Dimension checks
    g.null_weights,
    g.non_positive_weights,
    g.null_dimensions,
    g.non_positive_dimensions,
    g.null_volumes,
    g.non_positive_volumes,
    g.negative_photo_counts,
    g.invalid_weight_categories,
    g.invalid_size_categories,

    -- Sanity metrics
    g.avg_weight_kg,
    g.max_weight_kg,
    g.avg_volume_cm3,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != s.silver_row_count     THEN 'FAIL: Row count mismatch vs silver'
        WHEN g.duplicate_product_ids > 0                THEN 'FAIL: Duplicate product_ids'
        WHEN g.null_product_ids > 0                     THEN 'FAIL: NULL product_ids'
        WHEN g.non_positive_weights > 0                 THEN 'FAIL: Non-positive product weights'
        WHEN g.non_positive_dimensions > 0              THEN 'FAIL: Non-positive product dimensions'
        WHEN g.non_positive_volumes > 0                 THEN 'FAIL: Non-positive product volumes'
        WHEN g.invalid_weight_categories > 0            THEN 'FAIL: Invalid weight_category values'
        WHEN g.invalid_size_categories > 0              THEN 'FAIL: Invalid size_category values'
        WHEN g.negative_photo_counts > 0                THEN 'FAIL: Negative photo quantities'
        WHEN g.unknown_categories > 0                   THEN 'WARN: Products with unknown category'
        WHEN g.null_weights > 0                         THEN 'WARN: Products with NULL weight'
        WHEN g.null_dimensions > 0                      THEN 'WARN: Products with NULL dimensions'
        ELSE                                                 'PASS'
    END AS validation_result

FROM silver_counts s, gold_counts g;

silver_row_count,gold_row_count,row_difference,distinct_product_ids,duplicate_product_ids,null_product_ids,null_categories,unknown_categories,distinct_categories,null_weights,non_positive_weights,null_dimensions,non_positive_dimensions,null_volumes,non_positive_volumes,negative_photo_counts,invalid_weight_categories,invalid_size_categories,avg_weight_kg,max_weight_kg,avg_volume_cm3,validation_result
32951,32951,0,32951,0,0,0,610,14,6,0,2,0,2,0,0,0,0,2.277,40.425,16564.1,WARN: Products with unknown category


In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_sellers AS

SELECT
    -- Keys
    s.seller_id,
    --Attributes
    s.seller_city_cleaned AS seller_city,
    s.seller_state,
    s.seller_region,
    s.seller_zip_code_prefix AS seller_zip_code,
    AVG(g.geolocation_lat_updated) AS latitude,
    AVG(g.geolocation_lng_updated) AS longitude,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_TIMESTAMP() AS updated_at
FROM silver_sellers s  
LEFT JOIN silver_geolocation g ON
    s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
GROUP BY
    s.seller_id,
    s.seller_city_cleaned,
    s.seller_state,
    s.seller_region,
    s.seller_zip_code_prefix;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH silver_counts AS (
    SELECT COUNT(*) AS silver_row_count
    FROM silver_sellers
),
gold_counts AS (
    SELECT
        COUNT(*)                                                            AS gold_row_count,
        COUNT(DISTINCT seller_id)                                           AS distinct_seller_ids,
        COUNT(*) - COUNT(DISTINCT seller_id)                                AS duplicate_seller_ids,
        SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END)                 AS null_seller_ids,
        SUM(CASE WHEN seller_city IS NULL THEN 1 ELSE 0 END)               AS null_cities,
        SUM(CASE WHEN seller_state IS NULL THEN 1 ELSE 0 END)              AS null_states,
        SUM(CASE WHEN seller_region IS NULL THEN 1 ELSE 0 END)             AS null_regions,
        SUM(CASE WHEN seller_zip_code IS NULL THEN 1 ELSE 0 END)           AS null_zip_codes,
        SUM(CASE WHEN seller_zip_code < 1000
                  OR seller_zip_code > 99999 THEN 1 ELSE 0 END)            AS invalid_zip_codes,
        SUM(CASE WHEN seller_state NOT IN (
                'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA',
                'MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN',
                'RS','RO','RR','SC','SP','SE','TO')
                THEN 1 ELSE 0 END)                                          AS invalid_states,
        SUM(CASE WHEN seller_region NOT IN (
                'North', 'Northeast', 'Central-West', 'Southeast', 'South')
                THEN 1 ELSE 0 END)                                          AS invalid_regions,
        SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END)                  AS null_latitudes,
        SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END)                 AS null_longitudes,
        SUM(CASE WHEN latitude < -33.75
                  OR latitude > 5.27 THEN 1 ELSE 0 END)                    AS out_of_range_latitudes,
        SUM(CASE WHEN longitude < -73.99
                  OR longitude > -28.85 THEN 1 ELSE 0 END)                 AS out_of_range_longitudes,
        COUNT(DISTINCT seller_state)                                        AS distinct_states,
        COUNT(DISTINCT seller_region)                                       AS distinct_regions
    FROM gold_dim_sellers
)
SELECT
    -- Row parity
    s.silver_row_count,
    g.gold_row_count,
    g.gold_row_count - s.silver_row_count                                  AS row_difference,

    -- Key integrity
    g.distinct_seller_ids,
    g.duplicate_seller_ids,
    g.null_seller_ids,

    -- NULL checks
    g.null_cities,
    g.null_states,
    g.null_regions,
    g.null_zip_codes,

    -- Domain validation
    g.invalid_zip_codes,
    g.invalid_states,
    g.invalid_regions,

    -- Geolocation checks
    g.null_latitudes,
    g.null_longitudes,
    g.out_of_range_latitudes,
    g.out_of_range_longitudes,

    -- Sanity metrics
    g.distinct_states,
    g.distinct_regions,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != s.silver_row_count     THEN 'FAIL: Row count mismatch vs silver'
        WHEN g.duplicate_seller_ids > 0                 THEN 'FAIL: Duplicate seller_ids'
        WHEN g.null_seller_ids > 0                      THEN 'FAIL: NULL seller_ids'
        WHEN g.invalid_states > 0                       THEN 'FAIL: Invalid Brazilian state codes'
        WHEN g.invalid_regions > 0                      THEN 'FAIL: Invalid region values'
        WHEN g.invalid_zip_codes > 0                    THEN 'FAIL: Zip codes outside valid range'
        WHEN g.out_of_range_latitudes > 0               THEN 'WARN: Latitudes outside Brazilian bounds'
        WHEN g.out_of_range_longitudes > 0              THEN 'WARN: Longitudes outside Brazilian bounds'
        WHEN g.null_latitudes > 0                       THEN 'WARN: Sellers with no latitude (zip not in geolocation)'
        WHEN g.distinct_regions != 5                    THEN 'WARN: Expected 5 distinct regions'
        ELSE                                                 'PASS'
    END AS validation_result

FROM silver_counts s, gold_counts g;

silver_row_count,gold_row_count,row_difference,distinct_seller_ids,duplicate_seller_ids,null_seller_ids,null_cities,null_states,null_regions,null_zip_codes,invalid_zip_codes,invalid_states,invalid_regions,null_latitudes,null_longitudes,out_of_range_latitudes,out_of_range_longitudes,distinct_states,distinct_regions,validation_result
3095,3095,0,3095,0,0,0,0,0,0,0,0,0,7,7,0,0,23,5,WARN: Sellers with no latitude (zip not in geolocation)


In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_dates AS

WITH date_range AS (
    SELECT EXPLODE(SEQUENCE(
        TO_DATE('2016-01-01'),
        TO_DATE('2028-12-31'),
        INTERVAL 1 DAY
    )) AS date_value
)
SELECT 
    -- Primary Key
    DATE_FORMAT(date_value, 'yyyyMMdd') AS date_key,
    -- Date Components
    date_value AS full_date,
    YEAR(date_value) AS year,
    QUARTER(date_value) AS quarter,
    MONTH(date_value) AS month,
    DAY(date_value) AS day,
    DAYOFWEEK(date_value) AS day_of_week,
    DAYOFYEAR(date_value) AS day_of_year,
    WEEKOFYEAR(date_value) AS week_of_year,
    -- Month/Year Combinations
    DATE_FORMAT(date_value, 'yyyy-MM') AS year_month,
    -- Day Names
    DATE_FORMAT(date_value, 'EEEE') AS day_name,
    DATE_FORMAT(date_value, 'MMMM') AS month_name,
    -- Week Start/End (Sunday-Saturday)
    CAST(DATE_TRUNC('WEEK', date_value) AS DATE) AS week_start_date,
    DATE_ADD(DATE_TRUNC('WEEK', date_value), 6) AS week_end_date,
    -- Month Start/End
    CAST(DATE_TRUNC('MONTH', date_value) AS DATE) AS month_start_date,
    LAST_DAY(date_value) AS month_end_date,
    -- Business Flags
    CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN FALSE ELSE TRUE END AS is_weekday,
    CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
    -- Brazilian Holidays (add specific dates)
    CASE 
        WHEN DATE_FORMAT(date_value, 'MM-dd') = '01-01' THEN 'New Year''s Day'
        WHEN DATE_FORMAT(date_value, 'MM-dd') = '09-07' THEN 'Independence Day'
        WHEN DATE_FORMAT(date_value, 'MM-dd') = '11-15' THEN 'Republic Day'
        WHEN DATE_FORMAT(date_value, 'MM-dd') = '12-25' THEN 'Christmas'
        -- Add more Brazilian holidays
        ELSE NULL
    END AS holiday_name,
    CASE 
        WHEN DATE_FORMAT(date_value, 'MM-dd') IN ('01-01', '09-07', '11-15', '12-25')
        THEN TRUE ELSE FALSE END AS is_holiday,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_TIMESTAMP() AS updated_at
FROM date_range;

num_affected_rows,num_inserted_rows


In [0]:
%sql
WITH expected AS (
    -- Sequence spans 2016-01-01 to 2028-12-31 inclusive
    SELECT DATEDIFF(TO_DATE('2028-12-31'), TO_DATE('2016-01-01')) + 1 AS expected_row_count
),
gold_counts AS (
    SELECT
        COUNT(*)                                                            AS gold_row_count,
        COUNT(DISTINCT date_key)                                            AS distinct_date_keys,
        COUNT(*) - COUNT(DISTINCT date_key)                                 AS duplicate_date_keys,
        COUNT(DISTINCT full_date)                                           AS distinct_full_dates,
        SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END)                  AS null_date_keys,
        SUM(CASE WHEN full_date IS NULL THEN 1 ELSE 0 END)                 AS null_full_dates,
        SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END)                      AS null_years,
        SUM(CASE WHEN month IS NULL
                  OR month < 1
                  OR month > 12 THEN 1 ELSE 0 END)                         AS invalid_months,
        SUM(CASE WHEN quarter IS NULL
                  OR quarter < 1
                  OR quarter > 4 THEN 1 ELSE 0 END)                        AS invalid_quarters,
        SUM(CASE WHEN day IS NULL
                  OR day < 1
                  OR day > 31 THEN 1 ELSE 0 END)                           AS invalid_days,
        SUM(CASE WHEN day_of_week < 1
                  OR day_of_week > 7 THEN 1 ELSE 0 END)                    AS invalid_day_of_week,
        SUM(CASE WHEN is_weekday = TRUE
                  AND is_weekend = TRUE THEN 1 ELSE 0 END)                  AS conflicting_weekday_flags,
        SUM(CASE WHEN is_weekday = FALSE
                  AND is_weekend = FALSE THEN 1 ELSE 0 END)                 AS missing_weekday_flags,
        SUM(CASE WHEN month_start_date > full_date THEN 1 ELSE 0 END)      AS invalid_month_start,
        SUM(CASE WHEN month_end_date < full_date THEN 1 ELSE 0 END)        AS invalid_month_end,
        SUM(CASE WHEN week_start_date > full_date THEN 1 ELSE 0 END)       AS invalid_week_start,
        SUM(CASE WHEN week_end_date < full_date THEN 1 ELSE 0 END)         AS invalid_week_end,
        SUM(CASE WHEN is_holiday = TRUE THEN 1 ELSE 0 END)                 AS total_holidays,
        MIN(full_date)                                                      AS min_date,
        MAX(full_date)                                                      AS max_date,
        COUNT(DISTINCT year)                                                AS distinct_years,
        COUNT(DISTINCT year_month)                                          AS distinct_year_months
    FROM gold_dim_dates
)
SELECT
    -- Row parity
    e.expected_row_count,
    g.gold_row_count,
    g.gold_row_count - e.expected_row_count                                AS row_difference,

    -- Key integrity
    g.distinct_date_keys,
    g.duplicate_date_keys,
    g.distinct_full_dates,
    g.null_date_keys,
    g.null_full_dates,

    -- Component validity
    g.null_years,
    g.invalid_months,
    g.invalid_quarters,
    g.invalid_days,
    g.invalid_day_of_week,

    -- Flag consistency
    g.conflicting_weekday_flags,
    g.missing_weekday_flags,

    -- Boundary validity
    g.invalid_month_start,
    g.invalid_month_end,
    g.invalid_week_start,
    g.invalid_week_end,

    -- Sanity metrics
    g.total_holidays,
    g.min_date,
    g.max_date,
    g.distinct_years,
    g.distinct_year_months,

    -- PASS/FAIL
    CASE
        WHEN g.gold_row_count != e.expected_row_count   THEN 'FAIL: Row count does not match expected sequence length'
        WHEN g.duplicate_date_keys > 0                  THEN 'FAIL: Duplicate date_keys'
        WHEN g.null_date_keys > 0                       THEN 'FAIL: NULL date_keys'
        WHEN g.null_full_dates > 0                      THEN 'FAIL: NULL full_dates'
        WHEN g.invalid_months > 0                       THEN 'FAIL: Month values outside 1–12'
        WHEN g.invalid_quarters > 0                     THEN 'FAIL: Quarter values outside 1–4'
        WHEN g.invalid_days > 0                         THEN 'FAIL: Day values outside 1–31'
        WHEN g.conflicting_weekday_flags > 0            THEN 'FAIL: Dates flagged as both weekday and weekend'
        WHEN g.missing_weekday_flags > 0                THEN 'FAIL: Dates flagged as neither weekday nor weekend'
        WHEN g.invalid_month_start > 0                  THEN 'FAIL: month_start_date is after full_date'
        WHEN g.invalid_month_end > 0                    THEN 'FAIL: month_end_date is before full_date'
        WHEN g.distinct_years != 13                     THEN 'WARN: Expected 13 distinct years (2016–2028)'
        WHEN g.total_holidays = 0                       THEN 'WARN: No holidays found — check holiday logic'
        ELSE                                                 'PASS'
    END AS validation_result

FROM expected e, gold_counts g;

expected_row_count,gold_row_count,row_difference,distinct_date_keys,duplicate_date_keys,distinct_full_dates,null_date_keys,null_full_dates,null_years,invalid_months,invalid_quarters,invalid_days,invalid_day_of_week,conflicting_weekday_flags,missing_weekday_flags,invalid_month_start,invalid_month_end,invalid_week_start,invalid_week_end,total_holidays,min_date,max_date,distinct_years,distinct_year_months,validation_result
4749,4749,0,4749,0,4749,0,0,0,0,0,0,0,0,0,0,0,0,0,52,2016-01-01,2028-12-31,13,156,PASS


## Aggregate Tables

These aggregate tables pre-compute commonly queried business metrics from the fact and dimension tables above. Each table is grouped at a specific analytical grain: daily, monthly, by product, by seller, by customer, or by region. 
These are designed to power dashboards and reports without requiring end users to write complex JOIN or GROUP BY logic.

All aggregate tables in this layer:
- Source exclusively from gold fact and dimension tables
- Include date dimension columns (year, month, quarter, year_month) for direct time-series filtering without additional joins
- Are fully refreshed on each pipeline run via CREATE OR REPLACE TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_daily_sales AS

SELECT
    --Date Dimensions
    DATE(o.order_date_key) AS order_date,
    d.year,
    d.month,
    d.quarter,
    d.day_name,
    d.is_weekend,
    d.is_holiday,
    --Aggregated Metrics
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    ROUND(SUM(o.total_items), 2) AS total_items_sold,
    ROUND(SUM(o.order_total_value), 2) AS total_revenue,
    ROUND(SUM(o.order_freight), 2) AS total_freight,
    ROUND(AVG(o.order_total_value), 2) AS avg_order_value,
    ROUND(AVG(o.total_items), 2) AS avg_items_per_order,
    ROUND(AVG(o.review_score), 2) AS avg_order_review_score,
    --Delivery Metrics
    SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) AS orders_delivered_on_time,
    SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END) AS orders_delivered,
    SUM(CASE WHEN o.is_canceled THEN 1 ELSE 0 END) AS orders_canceled,
    AVG(o.days_to_deliver) AS avg_delivery_days,
    ROUND(SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
        2) AS on_time_delivery_rate_pct,
    ROUND(SUM(CASE WHEN o.is_canceled THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2) AS cancellation_rate_pct,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at
FROM gold_fact_orders o 
JOIN gold_dim_dates d ON 
    DATE(o.order_date_key) = d.full_date
GROUP BY
    order_date,
    d.year,
    d.month,
    d.quarter,
    d.day_name,
    d.is_weekend,
    d.is_holiday;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_monthly_sales AS

SELECT
    --Date Dimensions
    d.year,
    d.month,
    d.quarter,
    d.year_month,
    MIN(d.full_date) AS month_start_date,
    MAX(d.full_date) AS month_end_date,
    --Aggregated Metrics
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    ROUND(SUM(o.total_items), 2) AS total_items_sold,
    ROUND(SUM(o.order_total_value), 2) AS total_revenue,
    ROUND(SUM(o.order_freight), 2) AS total_freight,
    ROUND(AVG(o.order_total_value), 2) AS avg_order_value,
    ROUND(AVG(o.total_items), 2) AS avg_items_per_order,
    ROUND(AVG(o.review_score), 2) AS avg_order_review_score,
    --Delivery Metrics
    SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) AS orders_delivered_on_time,
    SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END) AS orders_delivered,
    SUM(CASE WHEN o.is_canceled THEN 1 ELSE 0 END) AS orders_canceled,
    AVG(o.days_to_deliver) AS avg_delivery_days,
    ROUND(SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
        2) AS on_time_delivery_rate_pct,
    ROUND(SUM(CASE WHEN o.is_canceled THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2) AS cancellation_rate_pct,
    --Monthly Shifts
    ROUND(SUM(o.order_total_value), 2) - LAG(ROUND(SUM(o.order_total_value), 2)) OVER (ORDER BY d.year, d.month) AS revenue_MOM_change,
    ROUND(
        (ROUND(SUM(o.order_total_value), 2) - LAG(ROUND(SUM(o.order_total_value), 2)) OVER (ORDER BY d.year, d.month)) * 100.0
        / NULLIF(LAG(ROUND(SUM(o.order_total_value), 2)) OVER (ORDER BY d.year, d.month), 0), 2) AS revenue_mom_pct_change,
    -- Audit
    CURRENT_TIMESTAMP() AS created_at
FROM gold_fact_orders o
JOIN gold_dim_dates d ON
    DATE(o.order_date_key) = d.full_date
GROUP BY
    d.year,
    d.month,
    d.quarter,
    d.year_month
ORDER BY
    d.year,
    d.month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_product_performance AS

SELECT
    --Date Dimensions
    d.year,
    d.month,
    d.quarter,
    d.year_month,
    --Dimensions
    p.product_id,
    p.product_category,
    p.weight_category,
    p.size_category,
    --Sales Metrics
    ROUND(SUM(oi.item_quantity), 2) AS total_units_sold,
    COUNT(DISTINCT oi.order_id) AS total_orders_including_item,
    ROUND(SUM(oi.item_total_value), 2) AS total_revenue,
    ROUND(AVG(oi.item_price), 2) AS avg_item_price,
    ROUND(AVG(oi.item_freight), 2) AS avg_freight_cost,
    ROUND(SUM(oi.item_total_value) / NULLIF(SUM(oi.item_quantity), 0), 2) AS avg_revenue_per_unit,
    --Review Metrics
    ROUND(AVG(o.review_score), 2) AS avg_review_score,
    COUNT(o.review_score) AS total_reviews,
    --Delivery Metrics
    ROUND(AVG(oi.days_to_deliver), 2) AS avg_days_to_deliver_item,
    SUM(CASE WHEN oi.delivery_performance = 'On Time' THEN 1 ELSE 0 END) AS on_time_deliveries,
    SUM(CASE WHEN oi.delivery_performance = 'Late' THEN 1 ELSE 0 END) AS late_deliveries,
    SUM(CASE WHEN oi.is_delivered THEN 1 ELSE 0 END) AS orders_with_item_delivered,
    SUM(CASE WHEN oi.is_canceled THEN 1 ELSE 0 END) AS orders_with_item_canceled,
    --Audit
    CURRENT_TIMESTAMP() AS created_at
FROM gold_fact_order_items oi
LEFT JOIN gold_dim_products p ON
    oi.product_id = p.product_id
LEFT JOIN gold_dim_dates d ON
    DATE(oi.order_purchase_date) = d.full_date
LEFT JOIN gold_fact_orders o ON
    oi.order_id = o.order_id
GROUP BY
    d.year,
    d.month,
    d.quarter,
    d.year_month,
    p.product_id,
    p.product_category,
    p.weight_category,
    p.size_category
ORDER BY
    d.year,
    d.month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_top_sellers AS

with aggregates AS (
    SELECT
        --Date Dimensions
        d.year,
        d.month,
        d.quarter,
        d.year_month,
        --Dimensions
        oi.seller_id,
        s.seller_city,
        s.seller_state,
        --Sales Metrics
        COUNT(DISTINCT oi.order_id) as total_orders_sold,
        COUNT(oi.product_id) as total_products_sold,
        ROUND(SUM(oi.item_total_value), 2) as total_revenue,
        SUM(CASE 
            WHEN oi.delivery_performance = 'Late' then 1
            ELSE 0
            END) * 1.0 / COUNT(*) AS late_delivery_rate
        FROM gold_fact_order_items oi
        JOIN gold_dim_dates d ON
            DATE(oi.order_purchase_date) = d.full_date
        JOIN gold_dim_sellers s ON
            oi.seller_id = s.seller_id  
        WHERE delivery_performance IS NOT NULL
        GROUP BY 
            d.year,
            d.month,
            d.quarter,
            d.year_month,
            oi.seller_id,
            s.seller_city,
            s.seller_state
        HAVING COUNT(DISTINCT oi.order_id) > 5
)
SELECT
    *,
    DENSE_RANK() over (order by total_revenue desc) as sales_ranking,
    PERCENT_RANK() OVER (ORDER BY total_revenue) AS percentile_rank,
    --Audit
    CURRENT_TIMESTAMP() AS created_at
FROM aggregates
ORDER BY seller_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_customer_lifespan AS

WITH aggregates AS (
    SELECT
        --Dimensions
        c.customer_unique_id AS customer_id,
        --Sales Metrics
        COUNT(DISTINCT oi.order_id) AS total_orders,
        COUNT(oi.product_id) AS total_items_ordered,
        ROUND(SUM(oi.item_total_value), 2) AS lifetime_spend,
        ROUND(AVG(oi.item_total_value), 2) AS avg_order_value,
        MIN(oi.order_purchase_date) AS first_purchase_date,
        MAX(oi.order_purchase_date) AS most_recent_purchase_date,
        DATEDIFF(MAX(oi.order_purchase_date), MIN(oi.order_purchase_date)) AS customer_lifespan_days,
        CASE
            WHEN COUNT(DISTINCT oi.order_id) = 1 THEN 'one_time'
            WHEN COUNT(DISTINCT oi.order_id) <= 2 THEN 'repeat'
            WHEN COUNT(DISTINCT oi.order_id) <= 5 THEN 'frequent'
            ELSE 'loyal'
            END AS lifecycle_stage
    FROM gold_fact_order_items oi
    JOIN gold_dim_customers c ON
        oi.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
    ORDER BY customer_lifespan_days DESC
)
SELECT
    *,
    DENSE_RANK() OVER (ORDER BY lifetime_spend DESC) AS total_spending_rank,
    --Audit
    CURRENT_TIMESTAMP() AS created_at
FROM aggregates
ORDER BY lifetime_spend DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_agg_regional_sales AS

SELECT
    --Date Dimensions
    d.year,
    d.month,
    d.quarter,
    d.year_month,
    --Dimensions
    c.customer_region,
    c.customer_state,
    --Sales Metrics
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.product_id) as total_items_ordered,
    COUNT(DISTINCT oi.seller_id) as total_sellers,
    ROUND(SUM(oi.item_total_value), 2) as total_revenue,
    ROUND(SUM(p.product_volume_cm3/1000), 2) as total_product_volume_litres,
    ROUND(SUM(p.product_weight_kg), 2) as total_product_weight_kg,
    --Audit
    CURRENT_TIMESTAMP() AS created_at
FROM gold_dim_customers c
JOIN gold_fact_orders o on c.customer_id = o.customer_id
JOIN gold_fact_order_items oi on oi.order_id = o.order_id
JOIN gold_dim_products p on p.product_id = oi.product_id
JOIN gold_dim_dates d on DATE(oi.order_purchase_date) = d.full_date
GROUP BY 
    c.customer_region,
    c.customer_state,
    d.year,
    d.month,
    d.quarter,
    d.year_month
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows
